# Dataset and DataLoader

Language models learn by predicting the next token. Given "The cat sat", the model should predict "on". This notebook shows how to prepare training data in this format.

**What we'll cover:**
- The sliding window approach to creating training pairs
- Building a PyTorch Dataset
- Creating batches with DataLoader

In [1]:
import os
import requests
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.12.0


In [2]:
# Load our text
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    response = requests.get(url, timeout=30)
    with open("the-verdict.txt", "wb") as f:
        f.write(response.content)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
enc_text = tokenizer.encode(raw_text)

print(f"Total tokens: {len(enc_text)}")

Total tokens: 5145


## The Input-Target Relationship

For language modeling, the target is simply the input shifted by one position. If our context window is 4 tokens:

- Input:  `[token_0, token_1, token_2, token_3]`
- Target: `[token_1, token_2, token_3, token_4]`

The model learns to predict each token from all the tokens before it.

In [3]:
# Let's look at a small example
enc_sample = enc_text[50:]
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f"Input:  {x}")
print(f"Target: {y}")

Input:  [290, 4920, 2241, 287]
Target: [4920, 2241, 287, 257]


In [4]:
# What the model learns at each position
print("Position-by-position predictions:\n")

for i in range(1, context_size + 1):
    context = enc_sample[:i]
    target = enc_sample[i]
    
    context_text = tokenizer.decode(context)
    target_text = tokenizer.decode([target])
    
    print(f"'{context_text}' -> '{target_text}'")

Position-by-position predictions:

' and' -> ' established'
' and established' -> ' himself'
' and established himself' -> ' in'
' and established himself in' -> ' a'


## Sliding Window

To create multiple training examples from one text, we slide a window across the tokens. The `stride` parameter controls how much the window moves each step:

- `stride = 1`: Maximum overlap, most examples, slower training
- `stride = context_size`: No overlap, fewer examples, faster training

## Building the Dataset

PyTorch's Dataset class needs two methods:
- `__len__()`: How many examples we have
- `__getitem__(idx)`: Return the example at index `idx`

In [5]:
class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        
        # Sliding window to create chunks
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

## Creating a DataLoader

The DataLoader handles batching, shuffling, and parallel loading. Here's a helper function that puts it all together.

In [6]:
def create_dataloader(txt, batch_size=4, max_length=256, stride=128,
                      shuffle=True, drop_last=True, num_workers=0):
    
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    
    return dataloader

## Testing with Small Batches

Let's see what the data looks like with a small context size.

In [7]:
# Small example: batch_size=1, context=4, stride=1
dataloader = create_dataloader(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)

first_batch = next(data_iter)
print("First batch:")
print(f"  Input:  {first_batch[0]}")
print(f"  Target: {first_batch[1]}")

First batch:
  Input:  tensor([[  40,  367, 2885, 1464]])
  Target: tensor([[ 367, 2885, 1464, 1807]])


In [8]:
second_batch = next(data_iter)
print("\nSecond batch:")
print(f"  Input:  {second_batch[0]}")
print(f"  Target: {second_batch[1]}")

print("\nNotice: with stride=1, batches overlap by 3 tokens")


Second batch:
  Input:  tensor([[ 367, 2885, 1464, 1807]])
  Target: tensor([[2885, 1464, 1807, 3619]])

Notice: with stride=1, batches overlap by 3 tokens


## Larger Batches

In practice, we use larger batches and no overlap (stride = context size).

In [9]:
dataloader = create_dataloader(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs shape:", inputs.shape)
print("Targets shape:", targets.shape)

print("\nInputs:")
print(inputs)

print("\nTargets:")
print(targets)

Inputs shape: torch.Size([8, 4])
Targets shape: torch.Size([8, 4])

Inputs:
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Summary

We now have:
1. A Dataset that creates input-target pairs using a sliding window
2. A DataLoader that batches these pairs for training

Each input sequence of length N is paired with a target sequence that's shifted by one position. The model learns to predict the next token at every position.

Next: Token embeddings - converting token IDs into vectors the model can work with.